In [ ]:
"""
EEGNet Training Pipeline for Live-Recorded EEG Data
=====================================================
- Upload your ZIP file via the interface below
- Paths are auto-detected from the ZIP contents
- REST (Eyes-Closed) = label 0  |  TASK = label 1
- Balanced sampling: equal epochs from each class
Architecture : EEGNet-8,2  (Lawhern et al., 2018)
"""

# ════════════════════════════════════════════════════════
# STEP 1 — Auto-install all required packages
# ════════════════════════════════════════════════════════
import subprocess, sys

REQUIRED_PKGS = {
    "mne":          "mne",
    "sklearn":      "scikit-learn",
    "torch":        "torch",
    "numpy":        "numpy",
    "scipy":        "scipy",
    "matplotlib":   "matplotlib",
    "seaborn":      "seaborn",
}

print("Checking dependencies...")
for import_name, install_name in REQUIRED_PKGS.items():
    try:
        __import__(import_name)
    except ImportError:
        print(f"  Installing {install_name}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", install_name],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        print(f"  ✓ {install_name} installed")
print("All dependencies ready.\n")


# ════════════════════════════════════════════════════════
# STEP 2 — Imports
# ════════════════════════════════════════════════════════
import os, random, warnings, zipfile, glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import mne

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

# ════════════════════════════════════════════════════════
# STEP 3 — Upload ZIP file
# ════════════════════════════════════════════════════════
print("=" * 60)
print("  UPLOAD YOUR EEG ZIP FILE")
print("=" * 60)
print("Your ZIP must contain:")
print("  • A folder with 'Eyes_Closed' or 'Rest' or 'EC' subfolders → label 0 (REST)")
print("  • A folder with 'Task' or 'TASK' subfolders               → label 1 (TASK)")
print("  • EDF files (.edf) inside those subfolders")
print()

try:
    from google.colab import files as colab_files
    print("Running in Google Colab — using file upload widget...")
    uploaded = colab_files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded. Please re-run and upload your ZIP.")
    zip_path = list(uploaded.keys())[0]
    print(f"Uploaded: {zip_path}")
except ImportError:
    # Not in Colab — look for ZIP in current directory or prompt path
    zips = glob.glob("*.zip")
    if zips:
        zip_path = zips[0]
        print(f"Found ZIP in current directory: {zip_path}")
    else:
        zip_path = input("Enter full path to your EEG ZIP file: ").strip().strip('"')
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"File not found: {zip_path}")

# ════════════════════════════════════════════════════════
# STEP 4 — Extract ZIP and auto-detect EDF file paths
# ════════════════════════════════════════════════════════
EXTRACT_DIR = "eeg_data_extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

print(f"\nExtracting ZIP to '{EXTRACT_DIR}/'...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(EXTRACT_DIR)
print("Extraction complete.\n")

# Find all EDF files recursively
all_edfs = glob.glob(os.path.join(EXTRACT_DIR, "**", "*.edf"), recursive=True)
all_edfs += glob.glob(os.path.join(EXTRACT_DIR, "**", "*.EDF"), recursive=True)

if not all_edfs:
    raise FileNotFoundError(
        f"No .edf files found after extracting ZIP.\n"
        f"Contents of {EXTRACT_DIR}:\n" +
        "\n".join(os.listdir(EXTRACT_DIR))
    )

print(f"Found {len(all_edfs)} EDF file(s):")
for f in all_edfs:
    print(f"  {f}")

# Auto-classify EDF files into REST vs TASK by folder name keywords
REST_KEYWORDS = ["eyes_closed", "ec", "rest", "baseline", "resting"]
TASK_KEYWORDS = ["task", "active", "motor", "mental", "cognitive"]

EC_FILES   = []
TASK_FILES = []

for edf in all_edfs:
    folder = os.path.dirname(edf).lower().replace("\\", "/")
    fname  = os.path.basename(edf).lower()
    combined = folder + "/" + fname

    is_rest = any(kw in combined for kw in REST_KEYWORDS)
    is_task = any(kw in combined for kw in TASK_KEYWORDS)

    if is_rest and not is_task:
        EC_FILES.append(edf)
    elif is_task and not is_rest:
        TASK_FILES.append(edf)
    else:
        # Ambiguous — show warning and skip
        print(f"  ⚠ Could not classify (skipping): {edf}")

print(f"\nAuto-classified:")
print(f"  REST files ({len(EC_FILES)}): {[os.path.basename(f) for f in EC_FILES]}")
print(f"  TASK files ({len(TASK_FILES)}): {[os.path.basename(f) for f in TASK_FILES]}")

if not EC_FILES:
    raise ValueError(
        "No REST/Eyes-Closed EDF files detected.\n"
        "Make sure your ZIP has a folder named 'Eyes_Closed', 'Rest', 'EC', or 'Baseline'."
    )
if not TASK_FILES:
    raise ValueError(
        "No TASK EDF files detected.\n"
        "Make sure your ZIP has a folder named 'Task', 'Active', or 'Motor'."
    )

# ════════════════════════════════════════════════════════
# STEP 5 — Config
# ════════════════════════════════════════════════════════
SEED        = 42
SFREQ       = None    # auto-detected from data
EPOCH_SEC   = 2.0     # seconds per epoch
STEP_SEC    = 1.0     # sliding window step (50% overlap)
L_FREQ      = 1.0     # band-pass low Hz
H_FREQ      = 40.0    # band-pass high Hz
N_EPOCHS_TR = 60      # training epochs
LR          = 1e-3
BATCH_SIZE  = 32
DROPOUT     = 0.5
OUTPUT_DIR  = "eegnet_outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {DEVICE}")

# ════════════════════════════════════════════════════════
# STEP 6 — Load & Epoch EDF files
# ════════════════════════════════════════════════════════
# Reference / non-brain channels to drop if present
DROP_CH_KEYWORDS = ["a1", "a2", "x1", "x2", "ekg", "ecg", "emg", "stim", "status"]

def auto_drop_channels(raw):
    """Drop known reference/non-EEG channels by name keyword."""
    to_drop = [ch for ch in raw.ch_names
               if any(kw in ch.lower() for kw in DROP_CH_KEYWORDS)]
    if to_drop:
        print(f"    Dropping non-EEG channels: {to_drop}")
        raw.drop_channels(to_drop)
    return raw

def load_and_epoch(paths, label, sfreq_ref):
    """
    Load EDF → band-pass filter → sliding window epochs.
    Returns (epochs array [N, C, T], detected_sfreq)
    """
    all_epochs = []
    detected_sfreq = sfreq_ref

    for path in paths:
        print(f"  Loading: {os.path.basename(path)}")
        try:
            raw = mne.io.read_raw_edf(path, preload=True, verbose=False)
        except Exception as e:
            print(f"    ✗ Failed to load: {e} — skipping")
            continue

        raw = auto_drop_channels(raw)

        # Use the file's actual sampling frequency
        file_sfreq = raw.info['sfreq']
        if detected_sfreq is None:
            detected_sfreq = file_sfreq
        if file_sfreq != detected_sfreq:
            print(f"    Resampling {file_sfreq}Hz → {detected_sfreq}Hz")
            raw.resample(detected_sfreq, verbose=False)

        raw.filter(L_FREQ, H_FREQ, fir_design='firwin', verbose=False)
        data = raw.get_data()  # (C, T)

        # Per-channel z-score normalization
        data = (data - data.mean(axis=1, keepdims=True)) / \
               (data.std(axis=1, keepdims=True) + 1e-8)

        epoch_len = int(EPOCH_SEC * detected_sfreq)
        step_len  = int(STEP_SEC  * detected_sfreq)
        n_times   = data.shape[1]
        starts    = list(range(0, n_times - epoch_len + 1, step_len))

        for s in starts:
            all_epochs.append(data[:, s:s + epoch_len])

        print(f"    ✓ {len(starts)} epochs extracted  "
              f"({data.shape[0]} ch, {file_sfreq}Hz, {n_times/file_sfreq:.1f}s)  label={label}")

    if not all_epochs:
        raise ValueError(f"No valid epochs from label={label} files.")

    return np.array(all_epochs, dtype=np.float32), detected_sfreq

print("\n── Loading REST (Eyes-Closed) ──")
X_rest, SFREQ = load_and_epoch(EC_FILES,   label=0, sfreq_ref=SFREQ)

print("\n── Loading TASK ──")
X_task, SFREQ = load_and_epoch(TASK_FILES, label=1, sfreq_ref=SFREQ)

print(f"\nRaw epoch counts →  REST: {len(X_rest)}  |  TASK: {len(X_task)}")

# Verify channel counts match
if X_rest.shape[1] != X_task.shape[1]:
    raise ValueError(
        f"Channel mismatch: REST has {X_rest.shape[1]} ch, "
        f"TASK has {X_task.shape[1]} ch.\n"
        "Both classes must have the same EEG montage."
    )

# ════════════════════════════════════════════════════════
# STEP 7 — Balanced Sampling (key fix for your dataset)
# ════════════════════════════════════════════════════════
# Problem : REST recordings are longer → 2-4× more epochs → biased model
# Solution: Randomly sample min(N_rest, N_task) epochs from each class
n_min = min(len(X_rest), len(X_task))
print(f"\nBalancing → using {n_min} epochs per class  (total {n_min * 2})")
print(f"  Discarded  REST epochs : {len(X_rest) - n_min}  (surplus from longer recording)")
print(f"  Discarded  TASK epochs : {len(X_task) - n_min}")

rng      = np.random.default_rng(SEED)
idx_rest = rng.choice(len(X_rest), n_min, replace=False)
idx_task = rng.choice(len(X_task), n_min, replace=False)

X_bal = np.concatenate([X_rest[idx_rest], X_task[idx_task]], axis=0)
y_bal = np.array([0] * n_min + [1] * n_min, dtype=np.int64)

# Shape for EEGNet: (N, 1, C, T)
X_bal = X_bal[:, np.newaxis, :, :]

print(f"\nFinal tensor shape : {X_bal.shape}")
print(f"Class distribution : REST={np.sum(y_bal==0)}  TASK={np.sum(y_bal==1)}")

# ════════════════════════════════════════════════════════
# STEP 8 — Train / Val / Test split
# ════════════════════════════════════════════════════════
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_bal, y_bal, test_size=0.30, stratify=y_bal, random_state=SEED
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED
)
print(f"\nSplit → Train: {len(X_tr)}  |  Val: {len(X_val)}  |  Test: {len(X_te)}")

def to_loader(X, y, shuffle):
    ds = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

train_loader = to_loader(X_tr,  y_tr,  shuffle=True)
val_loader   = to_loader(X_val, y_val, shuffle=False)
test_loader  = to_loader(X_te,  y_te,  shuffle=False)

# ════════════════════════════════════════════════════════
# STEP 9 — EEGNet Model Definition
# ════════════════════════════════════════════════════════
class EEGNet(nn.Module):
    """
    EEGNet-8,2  (Lawhern et al., J. Neural Eng. 2018)
    Compact depthwise/separable CNN for BCI — works well on small datasets.

    Input  : (batch, 1, n_channels, n_times)
    Output : (batch, n_classes)
    """
    def __init__(self, n_classes, n_channels, n_times,
                 F1=8, D=2, dropout=0.5):
        super().__init__()
        F2 = F1 * D
        kern_temporal = int(SFREQ // 2)   # 0.5s temporal kernel
        pad_temporal  = kern_temporal // 2

        # ── Block 1 : Temporal conv + Depthwise spatial conv ──
        self.block1 = nn.Sequential(
            nn.Conv2d(1, F1,
                      kernel_size=(1, kern_temporal),
                      padding=(0, pad_temporal),
                      bias=False),
            nn.BatchNorm2d(F1),
            # Depthwise: one filter per channel → learns spatial patterns
            nn.Conv2d(F1, F2,
                      kernel_size=(n_channels, 1),
                      groups=F1,
                      bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        # ── Block 2 : Separable conv (depthwise + pointwise) ──
        self.block2 = nn.Sequential(
            nn.Conv2d(F2, F2, kernel_size=(1, 16), padding=(0, 8),
                      groups=F2, bias=False),          # depthwise
            nn.Conv2d(F2, F2, kernel_size=(1, 1), bias=False),  # pointwise
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(dropout),
        )

        # Auto-compute flattened size
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_times)
            flat  = self.block2(self.block1(dummy))
            flat_size = int(np.prod(flat.shape[1:]))

        self.classifier = nn.Linear(flat_size, n_classes)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


N_CH  = X_bal.shape[2]
N_T   = X_bal.shape[3]
model = EEGNet(n_classes=2, n_channels=N_CH, n_times=N_T,
               F1=8, D=2, dropout=DROPOUT).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nEEGNet  |  Input: (N, 1, {N_CH}, {N_T})  |  Params: {n_params:,}")

# ════════════════════════════════════════════════════════
# STEP 10 — Training
# ════════════════════════════════════════════════════════
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS_TR)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = n = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            loss   = criterion(logits, yb)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(yb)
            correct    += (logits.argmax(1) == yb).sum().item()
            n          += len(yb)
    return total_loss / n, correct / n

history  = {"tr_loss": [], "tr_acc": [], "val_loss": [], "val_acc": []}
best_val = 0.0
best_wts = None

print(f"\n{'Epoch':>5}  {'Tr-Loss':>8}  {'Tr-Acc':>7}  {'Val-Loss':>9}  {'Val-Acc':>8}")
print("─" * 52)

for ep in range(1, N_EPOCHS_TR + 1):
    tl, ta = run_epoch(train_loader, train=True)
    vl, va = run_epoch(val_loader,   train=False)
    scheduler.step()

    history["tr_loss"].append(tl);  history["tr_acc"].append(ta)
    history["val_loss"].append(vl); history["val_acc"].append(va)

    if va > best_val:
        best_val = va
        best_wts = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if ep % 5 == 0 or ep == 1:
        print(f"{ep:>5}  {tl:>8.4f}  {ta:>7.3f}  {vl:>9.4f}  {va:>8.3f}")

# ════════════════════════════════════════════════════════
# STEP 11 — Evaluation on Test Set
# ════════════════════════════════════════════════════════
model.load_state_dict(best_wts)
model.eval()

all_preds, all_probs, all_true = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        logits = model(Xb.to(DEVICE))
        probs  = torch.softmax(logits, 1)[:, 1].cpu().numpy()
        preds  = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_probs.extend(probs)
        all_true.extend(yb.numpy())

acc    = accuracy_score(all_true, all_preds)
auc    = roc_auc_score(all_true, all_probs)
cm     = confusion_matrix(all_true, all_preds)
report = classification_report(all_true, all_preds,
                                target_names=["REST", "TASK"])

print(f"\n{'='*52}")
print(f"  Test Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Test ROC-AUC  : {auc:.4f}")
print(f"  Best Val Acc  : {best_val:.4f}")
print(f"{'='*52}")
print("\nClassification Report:\n", report)

# ════════════════════════════════════════════════════════
# STEP 12 — Save Model & Plots
# ════════════════════════════════════════════════════════
model_path = os.path.join(OUTPUT_DIR, "eegnet_19ch.pt")
torch.save({
    "model_state": best_wts,
    "n_channels":  N_CH,
    "n_times":     N_T,
    "sfreq":       SFREQ,
    "classes":     ["REST", "TASK"],
    "accuracy":    acc,
    "auc":         auc,
}, model_path)
print(f"\nModel saved → {model_path}")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("EEGNet Results (Balanced Sampling — REST vs TASK)", fontsize=13)
ep_x = range(1, N_EPOCHS_TR + 1)

axes[0].plot(ep_x, history["tr_loss"], label="Train", color="#2196F3")
axes[0].plot(ep_x, history["val_loss"], label="Val",   color="#FF5722")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep_x, history["tr_acc"], label="Train", color="#4CAF50")
axes[1].plot(ep_x, history["val_acc"], label="Val",   color="#FF9800")
axes[1].axhline(acc, ls="--", color="grey", label=f"Test {acc:.3f}")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch")
axes[1].legend(); axes[1].grid(alpha=0.3)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["REST","TASK"], yticklabels=["REST","TASK"],
            ax=axes[2], cbar=False)
axes[2].set_title(f"Confusion Matrix\nAcc={acc:.3f}  AUC={auc:.3f}")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("True")

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, "eegnet_results.png")
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot  saved → {plot_path}")

print("\n── Balance Summary ──────────────────────────────────")
print(f"  Total REST epochs (raw) : {len(X_rest)}")
print(f"  Total TASK epochs (raw) : {len(X_task)}")
print(f"  Epochs used per class   : {n_min}")
print(f"  ✓ Equal class contribution — no REST-bias in training")

Checking dependencies...
  Installing mne...
  ✓ mne installed
All dependencies ready.

  UPLOAD YOUR EEG ZIP FILE
Your ZIP must contain:
  • A folder with 'Eyes_Closed' or 'Rest' or 'EC' subfolders → label 0 (REST)
  • A folder with 'Task' or 'TASK' subfolders               → label 1 (TASK)
  • EDF files (.edf) inside those subfolders

Running in Google Colab — using file upload widget...


Saving labeled_EEG_data.zip to labeled_EEG_data.zip
Uploaded: labeled_EEG_data.zip

Extracting ZIP to 'eeg_data_extracted/'...
Extraction complete.

Found 3 EDF file(s):
  eeg_data_extracted/Madhav_Reddy_eeg_data/Eyes_Closed/Madhav reddy 01.000.02 AGE 21  EC.edf
  eeg_data_extracted/Madhav_Reddy_eeg_data/Eyes_Closed/Madhav reddy 01.000.04 AGE 21  EC.edf
  eeg_data_extracted/Madhav_Reddy_eeg_data/Task/Madhav reddy 01.000.03 AGE 21  TASK_1.edf

Auto-classified:
  REST files (2): ['Madhav reddy 01.000.02 AGE 21  EC.edf', 'Madhav reddy 01.000.04 AGE 21  EC.edf']
  TASK files (1): ['Madhav reddy 01.000.03 AGE 21  TASK_1.edf']

Device : cpu

── Loading REST (Eyes-Closed) ──
  Loading: Madhav reddy 01.000.02 AGE 21  EC.edf
    Dropping non-EEG channels: ['EEG A2-A1']
    ✓ 238 epochs extracted  (19 ch, 256.0Hz, 239.0s)  label=0
  Loading: Madhav reddy 01.000.04 AGE 21  EC.edf
    Dropping non-EEG channels: ['EEG A2-A1']
    ✓ 262 epochs extracted  (19 ch, 256.0Hz, 263.0s)  label=0

── Loading

In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "n_channels": 19,
    "n_times": 256,
    "sfreq": 128,
    "classes": ["No Stress", "Stress"]
}, "eegnet_19ch.pt")

In [ ]:
from google.colab import files
files.download("eegnet_19ch.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch, os

file = "/content/eegnet_19ch.pt"   # Colab
# file = "/kaggle/working/eegnet_64ch.pt"   # Kaggle

print(os.path.getsize(file))
ckpt = torch.load(file, map_location="cpu", weights_only=False)
print(ckpt.keys())

17488
dict_keys(['model_state', 'n_channels', 'n_times', 'sfreq', 'classes'])


In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "n_channels": 19,
    "n_times": 256,
    "sfreq": 128,
    "classes": ["No Stress", "Stress"]
}, "/content/eegnet_19ch.pt")

from google.colab import files
files.download("/content/eegnet_19ch.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
print(checkpoint.keys())

NameError: name 'checkpoint' is not defined